# 📖 Notebook 3: Right to Erasure (GDPR Article 17)

**Goal**: Implement the "right to be forgotten" — the most technically challenging GDPR requirement — with cascading deletes across both paired regions.

## Learning Objectives

By the end of this notebook, you'll understand:
- What GDPR Article 17 requires (right to erasure)
- Why deletion is harder than it sounds in distributed systems
- How to cascade deletes across related tables
- How to handle deletion across paired regions
- How to maintain audit trails even after data is deleted

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/gdpr-paired-regions
docker-compose up -d
```

### Visualization

- **Adminer**: http://localhost:8081  
  Watch users disappear from both region databases as we process erasure requests.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
from datetime import datetime

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 5434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

def get_connection(region):
    config = EU_WEST_CONFIG if region == "eu-west" else EU_NORTH_CONFIG
    return psycopg2.connect(**config)

# Verify setup
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ {region}: {cur.fetchone()[0]} users")
    conn.close()

✅ eu-west: 16 users
✅ eu-north: 16 users


## 1. What is the Right to Erasure?

**GDPR Article 17** says: EU citizens have the right to ask any company to **delete all their personal data**.

The company must:
1. Delete the data within **30 days** of the request
2. Delete it from **all systems** — databases, backups, caches, logs, analytics
3. Notify any **third parties** who received the data to delete it too
4. Confirm to the user that deletion is complete

### When Can You Refuse?

You can refuse erasure if:
- Data is needed for a **legal obligation** (e.g., tax records — typically 7 years)
- Data is needed for **public interest** (e.g., medical research)
- Data is needed for **legal claims** (e.g., active lawsuit)

### Why Is This Hard?

```
User asks: "Delete all my data"

You need to find and delete:
├── users table          → PII (name, email, phone)
├── addresses table      → PII (street, city)
├── orders table         → linked to user, contains purchase history
├── consent_log table    → paradox: proves consent, but contains PII
├── EU-West database     → primary copy
├── EU-North database    → replica copy
├── backups              → may contain the data
├── analytics systems    → user behavior data
└── third-party services → email provider, payment processor, etc.
```

Let's implement this step by step.

In [2]:
# ── First, let's see all the data we have for a specific user ──

def show_user_data_footprint(user_id, region):
    """Shows ALL data we have for a user across all tables."""
    conn = get_connection(region)
    cur = conn.cursor()

    print(f"\n📋 Data Footprint for User {user_id} in {region.upper()}")
    print("=" * 60)

    # Users table
    cur.execute("SELECT * FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()
    if user:
        print(f"\n👤 users table:")
        print(f"   Name: {user[2]}, Email: {user[1]}, Phone: {user[3]}")
        print(f"   DOB: {user[4]}, Country: {user[5]}, Region: {user[6]}")
    else:
        print("\n👤 users table: (no record)")

    # Addresses
    cur.execute("SELECT * FROM addresses WHERE user_id = %s", (user_id,))
    addresses = cur.fetchall()
    print(f"\n🏠 addresses table: {len(addresses)} record(s)")
    for addr in addresses:
        print(f"   {addr[2]}, {addr[3]} {addr[4]}, {addr[5]}")

    # Orders
    cur.execute("SELECT * FROM orders WHERE user_id = %s", (user_id,))
    orders = cur.fetchall()
    print(f"\n🛒 orders table: {len(orders)} record(s)")
    for order in orders:
        print(f"   {order[3]} — €{order[4]} ({order[6]})")

    # Consent log
    cur.execute("SELECT * FROM consent_log WHERE user_id = %s", (user_id,))
    consents = cur.fetchall()
    print(f"\n📜 consent_log table: {len(consents)} record(s)")
    for c in consents:
        print(f"   {c[2]}: {c[3]} (from IP {c[4]})")

    conn.close()


# Show data for Anna de Vries (user_id=1, EU-West)
show_user_data_footprint(1, "eu-west")


📋 Data Footprint for User 1 in EU-WEST

👤 users table:
   Name: Anna de Vries, Email: anna.devries@example.nl, Phone: +31-6-1234-5678
   DOB: 1990-03-15, Country: NL, Region: eu-west

🏠 addresses table: 1 record(s)
   Keizersgracht 42, Amsterdam 1015 CR, Netherlands

🛒 orders table: 2 record(s)
   299.99 — €EUR (2024-06-01 10:00:00)
   1299.00 — €EUR (2024-07-15 14:30:00)

📜 consent_log table: 3 record(s)
   granted: essential (from IP 82.168.1.100)
   granted: marketing (from IP 82.168.1.100)
   granted: analytics (from IP 82.168.1.100)


## 2. The Erasure Request Workflow

A proper GDPR erasure process looks like this:

```
User: "Delete my data"
       │
       ▼
  ┌─────────────┐
  │ Create       │ ← Log the request (BEFORE deleting anything)
  │ erasure      │
  │ request      │
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │ Check legal  │ ← Can we legally delete? (tax records, lawsuits)
  │ holds        │
  └──────┬──────┘
         │
    ┌────┴────┐
    ▼         ▼
  DELETE    DENY
  from all  with
  regions   reason
    │
    ▼
  ┌─────────────┐
  │ Confirm to   │
  │ user         │
  └─────────────┘
```

In [3]:
# ── Erasure Request System ─────────────────────────────────

# Tables that contain PII linked to a user.
# Order matters — delete from child tables first to respect
# foreign key constraints (even though we use ON DELETE CASCADE,
# explicit ordering is a best practice for auditability).
PII_TABLES = [
    "consent_log",   # child of users
    "orders",        # child of users
    "addresses",     # child of users
    "users",         # parent table — delete last
]


def submit_erasure_request(user_id, user_email, reason, region):
    """
    Step 1: Log the erasure request before deleting anything.
    GDPR requires proof that you received and processed the request.
    """
    conn = get_connection(region)
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO erasure_requests (user_id, user_email, reason, status)
        VALUES (%s, %s, %s, 'pending')
        RETURNING id
    """, (user_id, user_email, reason))

    request_id = cur.fetchone()[0]
    conn.commit()
    conn.close()

    print(f"📥 Erasure request #{request_id} created for user {user_id} ({user_email})")
    print(f"   Reason: {reason}")
    print(f"   Status: pending")
    return request_id


def check_legal_holds(user_id, region):
    """
    Step 2: Check if there are legal reasons we CANNOT delete this data.
    
    Common holds:
    - Tax records must be kept for 7 years in most EU countries
    - Active lawsuits require data preservation
    - Regulatory investigations
    """
    conn = get_connection(region)
    cur = conn.cursor()

    # Check for recent orders (simulate tax retention requirement)
    cur.execute("""
        SELECT COUNT(*) FROM orders
        WHERE user_id = %s
        AND created_at > NOW() - INTERVAL '7 years'
        AND status = 'completed'
    """, (user_id,))
    recent_orders = cur.fetchone()[0]
    conn.close()

    holds = []
    if recent_orders > 0:
        holds.append({
            "type": "tax_retention",
            "reason": f"User has {recent_orders} order(s) within the 7-year tax retention period.",
            "action": "Orders will be anonymized instead of fully deleted."
        })

    return holds


# Demo: Submit an erasure request for a specific user
print("📮 Submitting Erasure Request")
print("=" * 50)

# Let's use Anna de Vries (user_id=1) as our example
request_id = submit_erasure_request(
    user_id=1,
    user_email="anna.devries@example.nl",
    reason="User requested account deletion via privacy settings page",
    region="eu-west"
)

# Check legal holds
print("\n⚖️  Checking Legal Holds...")
holds = check_legal_holds(1, "eu-west")
if holds:
    for hold in holds:
        print(f"   ⚠️  Hold: {hold['type']}")
        print(f"      {hold['reason']}")
        print(f"      Action: {hold['action']}")
else:
    print("   ✅ No legal holds — full deletion allowed")

📮 Submitting Erasure Request
📥 Erasure request #1 created for user 1 (anna.devries@example.nl)
   Reason: User requested account deletion via privacy settings page
   Status: pending

⚖️  Checking Legal Holds...
   ⚠️  Hold: tax_retention
      User has 2 order(s) within the 7-year tax retention period.
      Action: Orders will be anonymized instead of fully deleted.


## 🚫 Bad → ✅ Best: Naive DELETE Leaves Orphans

Before building the proper erasure function, let's see why a naive one-line
`DELETE FROM users WHERE id = ?` is **not** GDPR-compliant.

### ❌ Bad: "just delete the user row"

If a schema has a child table **without** `ON DELETE CASCADE` (very common in
legacy systems), deleting only the parent row leaves "orphan" rows behind:

```
users (deleted)          addresses (still there!)
┌────┐                   ┌────┬─────────┬──────────────┐
│ id │                   │ id │ user_id │ street       │
├────┤                   ├────┼─────────┼──────────────┤
│ 99 │ ← deleted         │ 42 │   99    │ Privacy Lane │
└────┘                   └────┴─────────┴──────────────┘
                           ↑ orphan PII — GDPR violation!
```

The orphan rows still contain PII (a street address), but they now point at a
user that doesn't exist. The data is **still there**, just harder to find —
which is exactly what auditors look for.

### ✅ Best: cascade + explicit per-table deletes + audit log

That is what the `execute_erasure()` function below does:
1. Explicitly deletes from **every** child table (don't trust CASCADE alone).
2. Keeps a row in `erasure_requests` as proof the request was fulfilled.
3. Writes a `data_residency_log` entry for each table touched.


In [4]:
# ── Demo: the naive DELETE anti-pattern ─────────────────────
# We create a throw-away table WITHOUT cascade to prove the point.

conn = get_connection("eu-west")
cur = conn.cursor()

# Child table with NO foreign key, NO cascade — legacy-style
cur.execute("""
    CREATE TABLE IF NOT EXISTS legacy_addresses (
        id       SERIAL PRIMARY KEY,
        user_id  INTEGER,   -- no FK, no cascade (legacy schema)
        street   VARCHAR(255)
    )
""")

# Insert a demo user + a legacy-address row for them
cur.execute("""
    INSERT INTO users (email, full_name, country_code, home_region, consent_given)
    VALUES ('naive.demo@example.de', 'Naive Demo', 'DE', 'eu-west', TRUE)
    ON CONFLICT (email) DO UPDATE SET full_name = EXCLUDED.full_name
    RETURNING id
""")
demo_id = cur.fetchone()[0]
cur.execute("INSERT INTO legacy_addresses (user_id, street) VALUES (%s, %s)",
            (demo_id, "Privacy Lane 1"))
conn.commit()

# ❌ Naive deletion — only the parent row
cur.execute("DELETE FROM users WHERE id = %s", (demo_id,))
conn.commit()

# What remains?
cur.execute("SELECT COUNT(*) FROM users WHERE id = %s", (demo_id,))
users_left = cur.fetchone()[0]
cur.execute("SELECT COUNT(*), MAX(street) FROM legacy_addresses WHERE user_id = %s", (demo_id,))
orphans, street = cur.fetchone()

print("❌ After naive DELETE FROM users:")
print(f"   users rows remaining      : {users_left}  (good — user is gone)")
print(f"   legacy_addresses orphans  : {orphans}      (BAD — PII still there!)")
print(f"   leaked PII                : street = {street!r}")

# ✅ The fix: delete child rows too (or use ON DELETE CASCADE from the start)
cur.execute("DELETE FROM legacy_addresses WHERE user_id = %s", (demo_id,))
cur.execute("DROP TABLE legacy_addresses")   # cleanup demo table
conn.commit(); conn.close()

print("\n✅ Lesson: always enumerate *every* table with PII before erasing a user.")
print("   The execute_erasure() function below does exactly that.")


❌ After naive DELETE FROM users:
   users rows remaining      : 0  (good — user is gone)
   legacy_addresses orphans  : 1      (BAD — PII still there!)
   leaked PII                : street = 'Privacy Lane 1'

✅ Lesson: always enumerate *every* table with PII before erasing a user.
   The execute_erasure() function below does exactly that.


In [5]:
# ── The Core Erasure Function ──────────────────────────────

def execute_erasure(user_id, request_id, region):
    """
    Step 3: Actually delete (or anonymize) user data from a single region.
    
    This handles:
    - Cascading deletes across all PII tables
    - Anonymizing data that must be retained (tax records)
    - Logging every deletion for audit
    """
    conn = get_connection(region)
    cur = conn.cursor()
    deletion_report = []

    try:
        print(f"\n🗑️  Executing erasure for user {user_id} in {region.upper()}")
        print("-" * 50)

        # Check legal holds for this region
        holds = check_legal_holds(user_id, region)
        has_tax_hold = any(h["type"] == "tax_retention" for h in holds)

        # Step 3a: Delete consent logs (no legal requirement to keep)
        cur.execute("DELETE FROM consent_log WHERE user_id = %s", (user_id,))
        count = cur.rowcount
        deletion_report.append(("consent_log", count, "deleted"))
        print(f"   ✅ consent_log: {count} records deleted")

        # Step 3b: Handle orders (may need anonymization for tax)
        if has_tax_hold:
            # Anonymize instead of delete — keep financial data, remove PII
            cur.execute("""
                UPDATE orders
                SET user_id = NULL
                WHERE user_id = %s
            """, (user_id,))
            count = cur.rowcount
            deletion_report.append(("orders", count, "anonymized"))
            print(f"   ⚠️  orders: {count} records ANONYMIZED (tax retention hold)")
        else:
            cur.execute("DELETE FROM orders WHERE user_id = %s", (user_id,))
            count = cur.rowcount
            deletion_report.append(("orders", count, "deleted"))
            print(f"   ✅ orders: {count} records deleted")

        # Step 3c: Delete addresses
        cur.execute("DELETE FROM addresses WHERE user_id = %s", (user_id,))
        count = cur.rowcount
        deletion_report.append(("addresses", count, "deleted"))
        print(f"   ✅ addresses: {count} records deleted")

        # Step 3d: Delete the user record itself
        cur.execute("DELETE FROM users WHERE id = %s", (user_id,))
        count = cur.rowcount
        deletion_report.append(("users", count, "deleted"))
        print(f"   ✅ users: {count} records deleted")

        # Step 3e: Log the deletion in the residency log
        for table, count, action in deletion_report:
            if count > 0:
                cur.execute("""
                    INSERT INTO data_residency_log
                        (user_id, action, source_region, table_name, record_id, reason)
                    VALUES (%s, 'delete', %s, %s, %s,
                            'GDPR Article 17 erasure request #' || %s::text)
                """, (user_id, region, table, user_id, request_id))

        # Step 3f: Update the erasure request status
        cur.execute("""
            UPDATE erasure_requests
            SET status = 'completed', completed_at = NOW(), completed_by = 'system'
            WHERE id = %s
        """, (request_id,))

        conn.commit()
        print(f"\n   ✅ Erasure complete in {region}")
        return deletion_report

    except Exception as e:
        conn.rollback()
        print(f"   ❌ Erasure failed: {e}")
        raise
    finally:
        conn.close()


# Execute erasure in the primary region
print("🗑️  Processing Erasure Request")
print("=" * 50)
report = execute_erasure(user_id=1, request_id=request_id, region="eu-west")

🗑️  Processing Erasure Request

🗑️  Executing erasure for user 1 in EU-WEST
--------------------------------------------------
   ✅ consent_log: 3 records deleted
   ⚠️  orders: 2 records ANONYMIZED (tax retention hold)
   ✅ addresses: 1 records deleted
   ✅ users: 1 records deleted

   ✅ Erasure complete in eu-west


In [6]:
# ── Verify: Is the data really gone? ───────────────────────

print("🔍 Verification: Is Anna de Vries' data deleted?")
print("=" * 50)

show_user_data_footprint(1, "eu-west")

# Also check that the erasure request record still exists
# (this is the audit trail — it stays even after deletion)
conn = get_connection("eu-west")
cur = conn.cursor()
cur.execute("""
    SELECT id, user_email, reason, status, requested_at, completed_at
    FROM erasure_requests
    WHERE user_email = 'anna.devries@example.nl'
""")
request = cur.fetchone()
conn.close()

print("\n📋 Erasure Request Audit Trail (preserved):")
print(f"   Request #{request[0]}: {request[1]}")
print(f"   Reason: {request[2]}")
print(f"   Status: {request[3]}")
print(f"   Requested: {request[4]}")
print(f"   Completed: {request[5]}")
print("\n💡 The erasure request record itself is NOT deleted.")
print("   It's your proof that you fulfilled the GDPR request.")
print("   It only stores the email (which was provided in the request),")
print("   not any additional PII.")

🔍 Verification: Is Anna de Vries' data deleted?

📋 Data Footprint for User 1 in EU-WEST

👤 users table: (no record)

🏠 addresses table: 0 record(s)

🛒 orders table: 0 record(s)

📜 consent_log table: 0 record(s)



📋 Erasure Request Audit Trail (preserved):
   Request #1: anna.devries@example.nl
   Reason: User requested account deletion via privacy settings page
   Status: completed
   Requested: 2026-04-20 19:31:39.894711
   Completed: 2026-04-20 19:31:39.987640

💡 The erasure request record itself is NOT deleted.
   It's your proof that you fulfilled the GDPR request.
   It only stores the email (which was provided in the request),
   not any additional PII.


## 3. Cross-Region Erasure

If data was replicated to the paired region (as we did in Notebook 2), we must delete it there too. GDPR Article 17 requires deletion from **all** copies.

```
Erasure request received
        │
        ▼
  ┌─────────────┐    ┌─────────────┐
  │ Delete from  │    │ Delete from  │
  │ EU-West      │    │ EU-North     │
  │ (primary)    │    │ (replica)    │
  └──────┬──────┘    └──────┬──────┘
         │                   │
         └────────┬──────────┘
                  ▼
         Confirm to user:
         "All copies deleted"
```

In [7]:
# ── Full Cross-Region Erasure ──────────────────────────────

def full_gdpr_erasure(user_id, user_email, reason):
    """
    Complete GDPR Article 17 erasure across ALL paired regions.
    
    This is what a production system would do:
    1. Create erasure request in all regions
    2. Delete from primary region
    3. Delete from all replica regions
    4. Generate compliance report
    """
    all_regions = ["eu-west", "eu-north"]
    results = {}

    print(f"\n{'='*60}")
    print(f"🔒 GDPR ARTICLE 17 — FULL ERASURE PROCESS")
    print(f"   User: {user_email} (ID: {user_id})")
    print(f"   Reason: {reason}")
    print(f"{'='*60}")

    # Step 1: Create erasure requests in all regions
    request_ids = {}
    for region in all_regions:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO erasure_requests (user_id, user_email, reason, status)
            VALUES (%s, %s, %s, 'processing')
            RETURNING id
        """, (user_id, user_email, reason))
        request_ids[region] = cur.fetchone()[0]
        conn.commit()
        conn.close()
        print(f"\n📥 Erasure request #{request_ids[region]} created in {region}")

    # Step 2: Execute deletion in each region
    for region in all_regions:
        report = execute_erasure(user_id, request_ids[region], region)
        results[region] = report

    # Step 3: Generate summary
    print(f"\n{'='*60}")
    print("📊 ERASURE SUMMARY")
    print(f"{'='*60}")
    for region, report in results.items():
        print(f"\n  {region.upper()}:")
        for table, count, action in report:
            status = "✅" if action == "deleted" else "⚠️"
            print(f"    {status} {table}: {count} records {action}")

    print(f"\n✅ GDPR Article 17 erasure COMPLETE for {user_email}")
    print(f"   Data removed from {len(all_regions)} region(s)")
    print(f"   Audit trail preserved in erasure_requests table")


# Demo: Full erasure for Marc Dupont (user_id=2)
full_gdpr_erasure(
    user_id=2,
    user_email="marc.dupont@example.be",
    reason="User clicked 'Delete My Account' in account settings"
)


🔒 GDPR ARTICLE 17 — FULL ERASURE PROCESS
   User: marc.dupont@example.be (ID: 2)
   Reason: User clicked 'Delete My Account' in account settings

📥 Erasure request #2 created in eu-west



📥 Erasure request #1 created in eu-north

🗑️  Executing erasure for user 2 in EU-WEST
--------------------------------------------------
   ✅ consent_log: 2 records deleted
   ⚠️  orders: 1 records ANONYMIZED (tax retention hold)
   ✅ addresses: 1 records deleted
   ✅ users: 1 records deleted

   ✅ Erasure complete in eu-west

🗑️  Executing erasure for user 2 in EU-NORTH
--------------------------------------------------


   ✅ consent_log: 2 records deleted
   ⚠️  orders: 1 records ANONYMIZED (tax retention hold)
   ✅ addresses: 1 records deleted
   ✅ users: 1 records deleted

   ✅ Erasure complete in eu-north

📊 ERASURE SUMMARY

  EU-WEST:
    ✅ consent_log: 2 records deleted
    ⚠️ orders: 1 records anonymized
    ✅ addresses: 1 records deleted
    ✅ users: 1 records deleted

  EU-NORTH:
    ✅ consent_log: 2 records deleted
    ⚠️ orders: 1 records anonymized
    ✅ addresses: 1 records deleted
    ✅ users: 1 records deleted

✅ GDPR Article 17 erasure COMPLETE for marc.dupont@example.be
   Data removed from 2 region(s)
   Audit trail preserved in erasure_requests table


## 4. The Anonymization Pattern

Sometimes you can't fully delete data (legal holds), but you still need to satisfy GDPR. The solution is **anonymization** — removing all identifying information while keeping the statistical data.

| Before Anonymization | After Anonymization |
|---------------------|--------------------|
| Anna de Vries | [REDACTED] |
| anna@example.nl | user_12345@deleted |
| +31-6-1234-5678 | NULL |
| Keizersgracht 42 | NULL |
| Order: €299.99 | Order: €299.99 (user_id=NULL) |

The order amount is still useful for financial reporting, but it can no longer be linked to a person.

In [8]:
# ── Anonymization Demo ─────────────────────────────────────

def anonymize_user(user_id, region):
    """
    Anonymizes a user's PII while keeping non-identifying data.
    
    Use this when legal holds prevent full deletion.
    The user becomes statistically invisible but data is preserved
    for legal/financial reporting.
    """
    conn = get_connection(region)
    cur = conn.cursor()

    # Get user info before anonymization
    cur.execute("SELECT full_name, email FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()
    if not user:
        print(f"   User {user_id} not found in {region}")
        conn.close()
        return

    print(f"\n   Before: {user[0]} ({user[1]})")

    # Anonymize the user record
    anonymous_email = f"deleted_user_{user_id}@anonymized.local"
    cur.execute("""
        UPDATE users SET
            email = %s,
            full_name = '[REDACTED]',
            phone = NULL,
            date_of_birth = NULL,
            consent_given = FALSE,
            updated_at = NOW()
        WHERE id = %s
    """, (anonymous_email, user_id))

    # Anonymize addresses
    cur.execute("""
        UPDATE addresses SET
            street = '[REDACTED]',
            city = '[REDACTED]',
            postal_code = '[REDACTED]'
        WHERE user_id = %s
    """, (user_id,))
    addr_count = cur.rowcount

    # Delete consent logs (no need to keep)
    cur.execute("DELETE FROM consent_log WHERE user_id = %s", (user_id,))

    conn.commit()

    # Show the result
    cur.execute("SELECT full_name, email, phone, date_of_birth FROM users WHERE id = %s", (user_id,))
    anon = cur.fetchone()
    print(f"   After:  {anon[0]} ({anon[1]})")
    print(f"           Phone: {anon[2]}, DOB: {anon[3]}")
    print(f"   Addresses anonymized: {addr_count}")
    print(f"   ✅ User is now statistically invisible")

    conn.close()


print("🎭 Anonymization Demo")
print("=" * 50)
print("Scenario: Hans Müller has a tax retention hold on his orders.")
print("We can't delete his orders, so we anonymize his PII instead.")

anonymize_user(user_id=4, region="eu-west")

🎭 Anonymization Demo
Scenario: Hans Müller has a tax retention hold on his orders.
We can't delete his orders, so we anonymize his PII instead.

   Before: Hans Müller (hans.mueller@example.de)
   After:  [REDACTED] (deleted_user_4@anonymized.local)
           Phone: None, DOB: None
   Addresses anonymized: 1
   ✅ User is now statistically invisible


## 5. Why Microsoft Builds This Into Azure

### The Scale of the Problem

Microsoft processes erasure requests for:
- **Azure** (cloud infrastructure)
- **Microsoft 365** (email, documents, Teams)
- **LinkedIn** (professional profiles)
- **Xbox** (gaming accounts)
- **GitHub** (developer accounts)

Each service has PII scattered across **dozens of databases**, **multiple regions**, and **various backup systems**.

### Azure's Approach

1. **Azure Data Map** — automatically discovers and classifies PII across all services
2. **Azure Purview** — tracks where data flows (lineage)
3. **Compliance Manager** — scores your GDPR readiness
4. **Paired region deletion** — ensures erasure reaches all replicas

### Key Insight

The `ON DELETE CASCADE` constraint we used in our schema is a simple version of what Azure does at massive scale. When you delete a user, all related data across all tables is automatically cleaned up.

## 6. The Other Side of Erasure: Right to Access (Article 15)

Before you can *delete* a user's data, GDPR Article 15 says you must be able to
*export* it first. This is called a **Data Subject Access Request (DSAR)**:

> "The user has the right to obtain a copy of the personal data undergoing
> processing, in a **commonly used, machine-readable format**."

Article 20 ("data portability") goes further: users can ask for the export and
have it transmitted to another controller. So before shipping erasure, build
the export — the code is almost identical, you just `SELECT` instead of
`DELETE`.


In [9]:
# ── DSAR: Export all of a user's personal data as JSON ─────
import json

def export_user_data(user_id, region):
    """
    GDPR Article 15 / 20 — return *all* personal data we hold on the user
    in a machine-readable format (JSON). This is what you'd email the user
    when they submit a "download my data" request.
    """
    conn = get_connection(region)
    cur = conn.cursor()

    export = {"region": region, "generated_at": datetime.now().isoformat()}

    # Each query returns columns + rows so the output is self-describing.
    for table, sql in [
        ("users",       "SELECT * FROM users         WHERE id      = %s"),
        ("addresses",   "SELECT * FROM addresses     WHERE user_id = %s"),
        ("orders",      "SELECT * FROM orders        WHERE user_id = %s"),
        ("consent_log", "SELECT * FROM consent_log   WHERE user_id = %s"),
    ]:
        cur.execute(sql, (user_id,))
        cols = [d[0] for d in cur.description]
        export[table] = [dict(zip(cols, map(str, row))) for row in cur.fetchall()]

    conn.close()
    return export

# Demo: export everything we have on user 3 (Claire Martin, eu-west)
data = export_user_data(user_id=3, region="eu-west")
print("📤 DSAR export preview for user 3 (Claire Martin):")
print(json.dumps(data, indent=2, default=str)[:1200], "...\n")
print("💡 In production you would:")
print("   1. Verify the requester's identity (don't leak PII to impostors!)")
print("   2. Run this against *every* region and merge the results")
print("   3. Deliver as a signed, expiring download link")


📤 DSAR export preview for user 3 (Claire Martin):
{
  "region": "eu-west",
  "generated_at": "2026-04-20T22:31:40.221388",
  "users": [
    {
      "id": "3",
      "email": "claire.martin@example.fr",
      "full_name": "Claire Martin",
      "phone": "+33-1-4567-8901",
      "date_of_birth": "1992-11-08",
      "country_code": "FR",
      "home_region": "eu-west",
      "consent_given": "True",
      "consent_date": "2024-03-01 11:00:00",
      "created_at": "2026-04-20 19:31:16.310730",
      "updated_at": "2026-04-20 19:31:16.310730"
    }
  ],
  "addresses": [
    {
      "id": "3",
      "user_id": "3",
      "street": "Rue de Rivoli 75",
      "city": "Paris",
      "postal_code": "75001",
      "country": "France",
      "is_primary": "True",
      "created_at": "2026-04-20 19:31:16.311377"
    }
  ],
  "orders": [
    {
      "id": "4",
      "user_id": "3",
      "product": "Xbox Game Pass Ultimate",
      "amount": "14.99",
      "currency": "EUR",
      "status": "completed

## 🎯 Key Takeaways

1. **Right to erasure** (Art. 17) = EU citizens can demand full deletion of their data
2. **Naive `DELETE FROM users`** leaves orphan PII — enumerate *every* table
3. **Cascading deletes** must cover ALL tables with PII (use `ON DELETE CASCADE`)
4. **Cross-region deletion** — if data was replicated, delete from ALL regions
5. **Legal holds** may require anonymization instead of deletion
6. **Audit trail** — the erasure request record itself is kept as compliance proof
7. **30-day deadline** — GDPR gives you 30 days to complete the erasure
8. **Right to access** (Art. 15) — you must also be able to *export* the data on demand

## ⏭️ Next Up

In **Notebook 4**, we'll build a data sovereignty audit system — generating compliance reports that prove where data lives and how it moves between regions.
